### Give ratings for 10 seed movies, pick a target movie, and use a decision tree to predict the target rating!

In [57]:
# Imports 
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [58]:
# -------------------------------------------------
# Load MovieLens data
# -------------------------------------------------

ratings = pd.read_csv(
    "../ml-100k/u.data",
    sep="\t",
    names=[
        "user_id",
        "movie_id",
        "rating",
        "timestamp"
    ]
)

movies = pd.read_csv(
    "../ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0,1],
    names=[
        "movie_id",
        "title"
    ]
)
df = ratings.merge(
    movies,
    on="movie_id"
)

In [59]:
# Find top 10 movies with the most reviews
top_movies = (
    df.groupby(['movie_id', 'title'])
      .size()
      .reset_index(name='count')
      .sort_values('count', ascending=False)
)
# print top 10 movie titles
top_movies['title'].head(10)
for title in top_movies['title'].head(10):
    print (title)

Star Wars (1977)
Contact (1997)
Fargo (1996)
Return of the Jedi (1983)
Liar Liar (1997)
English Patient, The (1996)
Scream (1996)
Toy Story (1995)
Air Force One (1997)
Independence Day (ID4) (1996)


In [60]:
# -------------------------------------------------
# Select 10 movies for the user to rate
# -------------------------------------------------

# define seed movies as top 10 movies with most ratings
seed_movies = [title for title in top_movies['title'].head(10)]


user_ratings = {}

print("\nRate these movies from 1-5")
print("Enter 0 if you haven't seen it\n")


for title in seed_movies:

    movie = df[
        df["title"] == title
    ]

    if len(movie) == 0:
        continue

    movie_id = movie["movie_id"].iloc[0]

    rating = float(
        input(f"{title}: ")
    )

# create dictionary of user ratings
    user_ratings[movie_id] = rating



Rate these movies from 1-5
Enter 0 if you haven't seen it



In [61]:
seed_movies

['Star Wars (1977)',
 'Contact (1997)',
 'Fargo (1996)',
 'Return of the Jedi (1983)',
 'Liar Liar (1997)',
 'English Patient, The (1996)',
 'Scream (1996)',
 'Toy Story (1995)',
 'Air Force One (1997)',
 'Independence Day (ID4) (1996)']

In [62]:
# -------------------------------------------------
# Create only a 10 movie user-rating matrix
# -------------------------------------------------

seed_ids = list(user_ratings.keys())


comparison = (
    df[df["movie_id"].isin(seed_ids)]
    .pivot_table(
        index="user_id",
        columns="movie_id",
        values="rating"
    )
)

comparison = comparison[seed_ids]

In [63]:
# -------------------------------------------------
# Create user's rating vector
# -------------------------------------------------

# vector of user input ratings
user_vector = np.array(
    [
        user_ratings[movie_id]
        for movie_id in seed_ids
    ]
).reshape(1,-1)



tree = DecisionTreeRegressor(
    max_depth=15,
    random_state=42
)



In [64]:
# -------------------------------------------------
# Prepare training data: all ratings of seed movies
# -------------------------------------------------

# create df of just seed movies
seed_data = df[
    df["movie_id"].isin(seed_ids)
]

# Create 1 column df of ratings with seed movie ids for rows
X_train = seed_data[
    [
        "rating"
    ]
]


print(
    f"\nTraining data: {len(X_train)} ratings of seed movies"
)



tree = RandomForestRegressor(
    max_depth=15,
    random_state=42
)



Training data: 4863 ratings of seed movies


In [65]:
# -------------------------------------------------
# Predict a movie rating
# -------------------------------------------------

# take user target movie to predict
movie_choice = input(
    "\nEnter a movie title to predict: "
)

# find matching target movie in movies df
matches = movies[
    movies["title"]
    .str.contains(
        movie_choice,
        case=False,
        regex=False
    )
]

# Check if movie is in df and confirm
if len(matches) == 0:

    print("Movie not found")

else:

    print("\nPossible movies:")
    print(matches.head(10))


    matches = matches.reset_index(drop=True)

    # continue if only one movie is found, have user select one if multiple are found
    if len(matches) == 1:
        movie_id = int(matches.loc[0, "movie_id"])
        print(f"\nSelected movie: {matches.loc[0, 'title']}")
    else:
        print("\nMultiple matches found:")
        for idx, row in matches.iterrows():
            print(f"{idx + 1}. {row['title']}")

        selection = int(
            input(
                f"\nEnter selection number (1-{len(matches)}): "
            )
        )
        movie_id = int(matches.loc[selection - 1, "movie_id"])

    # select ratings of target movie, with user_id column and target_rating column
    target_ratings = df[
        df["movie_id"] == movie_id
    ][
        ["user_id", "rating"]
    ].rename(columns={"rating": "target_rating"})

    # create merged df with columns for seed movies and target movie
    train_data = comparison.merge(
        target_ratings,
        left_index=True,
        right_on="user_id"
    ).dropna()

    # seperate seed movies for X_train and target movie as y_train
    X_train = train_data[seed_ids]
    y_train = train_data["target_rating"]

    # print how many users have all seed movies and target movies
    print(
        f"\nTraining data: {len(X_train)} users with all seed movie ratings and target movie rating"
    )

    if len(X_train) == 0:
        print("Not enough training data for this movie.")
    else:
        tree.fit(
            X_train,
            y_train
        )

        # create test data of seed movie user inputs
        X_test = pd.DataFrame(
            user_vector,
            columns=seed_ids
        )

        # Give seed movie ratings to the model and predict target movie rating
        prediction = tree.predict(X_test)

        print(
            "\nPredicted rating:",
            round(prediction[0], 2)
        )


Possible movies:
    movie_id            title
26        27  Bad Boys (1995)

Selected movie: Bad Boys (1995)

Training data: 8 users with all seed movie ratings and target movie rating

Predicted rating: 3.48


In [66]:
target_ratings

,user_id,target_rating
1128,217,1
4001,1,2
8354,293,3
8637,16,2
14631,201,3
17109,13,3
18101,393,4
20273,452,5
20716,37,4
22457,268,4
